In [ ]:
import pandas as pd
import geopandas as gpd
import glob
from shapely.geometry import Point

# Path to the folder containing the CSV files
folder_path = './bikeshare_data/chicago_bikeshare'
# Use glob to find all CSV files in the folder
csv_files = glob.glob(f"{folder_path}/*.csv")

combined_df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
combined_df.columns

In [ ]:
chicago_stations = pd.read_csv('./bikeshare_data/chicago_bikeshare/Divvy_bicycle_stations_20250416.csv')
chicago_stations.columns
chicago_stations.shape

In [ ]:
start_stations = combined_df[['start_station_id', 
                     'start_lat', 
                     'start_lng']].drop_duplicates()

end_stations  = combined_df[['end_station_id', 
                     'end_lat', 
                     'end_lng']].drop_duplicates()

# Rename columns for uniform merging
end_stations.columns = ['Station ID', 'Station Latitude', 'Station Longitude']
start_stations.columns = ['Station ID', 'Station Latitude', 'Station Longitude']

# Combine start and end stations
all_stations = pd.concat([start_stations, end_stations]).drop_duplicates()
print(f'There are {len(all_stations)} unique stations in the Chicago system.')

all_stations_cleaned = all_stations.drop_duplicates(subset=['Station ID'])
print(f'There are {len(all_stations_cleaned)} unique stations in the Chicago system.')
all_stations_cleaned.columns

In [ ]:
trips_out = combined_df.groupby('start_station_id').size().reset_index(name='trips_out')


# Calculate trips into each station
trips_in = combined_df.groupby('end_station_id').size().reset_index(name='trips_in')

# Merge the trips_in and trips_out data
total_trips = pd.merge(trips_out, trips_in, 
                       left_on='start_station_id', 
                       right_on='end_station_id', 
                       how='outer', 
                       suffixes=('_out', '_in'))

# Calculate total trips for each station
total_trips['total_trips'] = total_trips['trips_out'].fillna(0) + total_trips['trips_in'].fillna(0)

# Rename the station ID column for consistency with stations_with_tracts
total_trips.rename(columns={'start_station_id': 'id'}, inplace=True)
columns = ['end_station_id', 'trips_in', 'trips_out']
total_trips = total_trips.drop(['end_station_id', 'trips_in', 'trips_out'], axis=1)

In [ ]:
tracts = gpd.read_file('./tract_shapefiles/tl_2023_il_tract/tl_2023_17_tract.shp')
# Create a geometry column from lat/long
geometry = [Point(xy) for xy in zip(all_stations_cleaned['Station Longitude'], 
                                    all_stations_cleaned['Station Latitude'])]
stations_gdf = gpd.GeoDataFrame(all_stations_cleaned, geometry=geometry, crs=tracts.crs)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,8))
tracts.plot(ax=ax, color='white', edgecolor='black')
stations_gdf.plot(ax=ax, marker='o', color='red', markersize=5)

plt.show()

# Spatial join to add tract geoids to each station
stations_with_tracts = gpd.sjoin(stations_gdf, tracts, how="left", predicate="within")
stations_with_tracts.rename(columns={'Station ID': 'id'}, inplace=True)
stations_with_tracts_and_trips = pd.merge(stations_with_tracts, total_trips, 
                                 on='id', 
                                 how='left')
tracts_with_trips = stations_with_tracts_and_trips.groupby('GEOID', as_index=False).agg({
    'total_trips': 'sum',
    'ALAND': 'first'})
# There are 821 days (inclusive) between January 1st, 2023 and March 31, 2025
tracts_with_trips['trips_per_day'] = tracts_with_trips['total_trips'] / 821
tracts_with_trips.to_csv('./bikeshare_data/tracts_with_trips/chicago_tracts_with_trips.csv')
tracts_with_trips.columns